# Order Book Imbalance Persistence
### An AR(1) study using local ETHUSD L2 snapshots

I investigate whether multi-level order book imbalance helps predict its next value.

**Workflow:** Question → Hypotheses → Experimental design → Testing → Conclusion.

Data: local Binance spot ETHUSD L2 message logs for 2026-10-10 to 2026-10-19 (`/Users/lujingyi/Desktop/样本`). The raw logs are incremental depth feeds, so `scripts/clean_l2.py` first rebuilds the book (sequence-checked) and writes one wide top-10 snapshot per update to `data/cleaned/`. This notebook reads only those cleaned files. Synthetic checks are software tests, not empirical evidence.

**Environment:** Python 3.9 (pandas, numpy, scipy, matplotlib) with `numpy`, `pandas`, `scipy`, `matplotlib`, and Jupyter. Install `ipywidgets` only for the optional interactive comparison. The notebook uses no exchange API or credentials.

## 1. Question

Does the current imbalance across the first K bid and ask levels predict the next sampled imbalance better than a training-mean baseline?

Predicting imbalance is not equivalent to predicting returns or trading profit. L2 contains price-level quantities, not necessarily individual order identities or queue positions.

## 2. Hypotheses

$$I_t=\frac{\sum_{i=1}^{K}Q^b_{t,i}-\sum_{i=1}^{K}Q^a_{t,i}}{\sum_{i=1}^{K}Q^b_{t,i}+\sum_{i=1}^{K}Q^a_{t,i}},\qquad I_{t+1}=c+\phi I_t+\varepsilon_{t+1}.$$

| Symbol | Meaning | Units / properties |
| --- | --- | --- |
| t | Integer sample index | Adjacent indices are one sampling interval apart |
| K | Number of price levels on each side | Positive integer; fixed before evaluation |
| Qᵇ, Qᵃ | Displayed quantities at a bid or ask level | Same quantity unit on both sides |
| I | Multi-level imbalance | Dimensionless; between −1 and +1 |
| c | Intercept | Dimensionless; estimated on training data |
| φ | Autoregressive coefficient | Dimensionless; estimated, not constrained by clipping |
| ε | Unexplained next-period imbalance | Dimensionless |

- **Dependence:** H0: φ = 0; H1: φ ≠ 0. A robust confidence interval describes uncertainty under the estimator's assumptions.
- **Adequacy:** residual serial correlation should be small if AR(1) captures the linear dependence. Residual ACF is a diagnostic, not proof of independence.
- **Prediction:** the model should reduce squared error on unseen observations. I also compare against persistence: predicting the next imbalance as the current one.

I do not assume normal innovations or stationary data merely because an AR(1) model is fitted. For bounded imbalance, linear predictions may leave [−1,1]; their frequency is reported rather than hidden.

## 3. Experimental design
### 3.1 Configure and inspect the actual data

The raw logs (`<date>.txt/<date>.txt`: one JSON message per line; snapshot, depth deltas, BBO, trades) are **not** read here. Run `python3 scripts/clean_l2.py` first. It reconstructs the book from the initial snapshot, rejects any update-id gap, checks for crossed books, cross-checks the rebuilt best levels against the BBO stream, clips each file to its UTC day, and writes `data/cleaned/eth_l2_<date>.csv` plus `cleaning_audit.json`. `binance_spot-ETHUSD-2026-10-10.csv` is excluded: it is byte-identical to the 2026-10-10 log.

Each cleaned row is the **complete top-10 book after one depth update**, stamped with the local receive time (`local_time_ns`, UTC nanoseconds). Price-change fields (`mid_price`, `spread`, `mid_change`) and trade files are kept for later work but are not used by this AR(1) study.

Start with a manageable training-only file. This first implementation loads selected columns into memory; it is not a streaming full-dataset pipeline.

In [ ]:
from pathlib import Path
import csv
import json
import hashlib
import platform
import warnings

import numpy as np
import pandas as pd
import scipy
from scipy.stats import norm
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    # Permit headless software checks outside a Jupyter environment.
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

# Cleaned snapshots produced by scripts/clean_l2.py from /Users/lujingyi/Desktop/样本.
CLEANED_DIR = Path("/Users/lujingyi/Downloads/time-series-analysis/projects/02_ar1_eth/data/cleaned")
DATA_FILES = sorted(CLEANED_DIR.glob("eth_l2_2026-10-*.csv"))
if not DATA_FILES:
    raise FileNotFoundError(f"No cleaned files in {CLEANED_DIR}; run scripts/clean_l2.py first")
SEPARATOR = ","
ENCODING = "utf-8-sig"

# Inspect the raw file before confirming its schema and timestamp meaning.
SNAPSHOT_SCHEMA_CONFIRMED = True  # Verified 2026-11-03: rebuilt book, 0 sequence gaps, 0 crossed, 0 BBO mismatches.
SYMBOL = "binance_spot:ETHUSD"  # From the log header (instrument id 2431025, tick 0.01, lot 0.0001).
TIMESTAMP_COLUMN = "local_time_ns"
TIMESTAMP_UNIT = "ns"
TIMESTAMP_MEANING = "local_receive_time"  # Recorder receive time; exchange_time_ns is kept in the file.

# Column names produced by scripts/clean_l2.py (10 levels stored; K levels used).
K = 5
COLUMN_TEMPLATES = {
    "bid_price": "bid_price_{level}", "bid_qty": "bid_qty_{level}",
    "ask_price": "ask_price_{level}", "ask_qty": "ask_qty_{level}",
}
# Fixed before viewing validation results. At 100 ms most adjacent grid points share one book state.
SAMPLE_INTERVAL = "1s"
# Delta feed: no update means no book change. Sequence ids are continuous and the longest
# silence is 254 s, so a book stays valid for up to 300 s after its last update.
MAX_SNAPSHOT_AGE = "300s"
TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
HAC_LAGS = 10  # Ten seconds on the 1 s grid; same physical span as the original 100 x 100 ms.
ACF_MAX_LAG = 20
# Forecast comparison (fixed before opening the test set): Diebold-Mariano on squared error,
# Bartlett HAC with 600 lags (10 min on the 1 s grid); 60 and 3600 lags reported as sensitivity.
DM_HAC_LAGS = 600
DM_SENSITIVITY_LAGS = (60, 3600)
DM_COMPARISONS = [("ar1", "mean"), ("ar1", "persistence")]  # (challenger, benchmark); first is primary
ALPHA = 0.05

# Inspect validation results first. Freeze choices before opening the final test.
RUN_FINAL_TEST = True  # Opened 2026-11-03 after freeze; see reports/2026-11-03_freeze_record.md
OUTPUT_DIR = Path("results/ar1_obi")

files = [Path(p).expanduser() for p in DATA_FILES]
for path in files:
    if not path.is_file():
        raise FileNotFoundError(f"Expected a real file: {path}")
if not files:
    print("No data configured. Set DATA_FILES, inspect the schema, then rerun.")
else:
    print({"first_file": str(files[0]), "bytes": files[0].stat().st_size})
    with files[0].open(encoding=ENCODING) as handle:
        for _ in range(3):
            print(handle.readline().rstrip()[:1000])
    preview = pd.read_csv(files[0], sep=SEPARATOR, encoding=ENCODING, nrows=5)
    display(preview)
    print("Columns:", preview.columns.tolist())

READY = bool(files) and SNAPSHOT_SCHEMA_CONFIRMED
if not READY:
    print("Analysis is gated until real files and a confirmed snapshot schema are available.")

### 3.2 Validate snapshots and construct imbalance

Prices must be positive, quantities nonnegative, bid levels strictly descending, and ask levels strictly ascending. Crossed books and zero total depth are rejected. Missing or invalid observations remain gaps; I never drop them and pretend the remaining rows are adjacent.

The timestamp unit is explicit. Naive ISO timestamps and conflicting snapshots at the same timestamp are rejected rather than silently resolved. If the source provides a reliable update sequence, adapt the tie rule after checking the format.

The conservative tie rule prevents unverified intra-timestamp ordering. Exact duplicate snapshots are counted and removed. Multiple symbols must be separated before loading this notebook.

In [ ]:
def parse_times(values, unit):
    """Convert explicit source timestamps to UTC without guessing their unit."""
    if unit in {"s", "ms", "us", "ns"}:
        numeric = pd.to_numeric(values, errors="raise")
        if numeric.isna().any() or not np.isfinite(numeric).all():
            raise ValueError("Missing or nonfinite timestamp")
        return pd.to_datetime(numeric, unit=unit, utc=True, errors="raise")
    if unit == "iso":
        stamps = values.map(pd.Timestamp)
        if any(pd.isna(t) or t.tzinfo is None for t in stamps):
            raise ValueError("ISO timestamps must include an explicit timezone")
        return pd.to_datetime(stamps, utc=True)
    raise ValueError("Set TIMESTAMP_UNIT explicitly after inspecting the source")


def normalize_snapshots(raw, timestamp_column, timestamp_unit, k, templates):
    """Validate wide snapshots; this function does not reconstruct delta feeds."""
    if not isinstance(k, int) or k < 1:
        raise ValueError("K must be a positive integer")
    columns = {side: [template.format(level=i) for i in range(1, k + 1)]
               for side, template in templates.items()}
    required = [timestamp_column] + [c for names in columns.values() for c in names]
    missing = sorted(set(required) - set(raw.columns))
    if missing:
        raise ValueError(f"Missing mapped columns: {missing}")
    data = raw[required].copy()
    data["time"] = parse_times(data[timestamp_column], timestamp_unit)
    price_qty_columns = [c for names in columns.values() for c in names]
    for name in price_qty_columns:
        data[name] = pd.to_numeric(data[name], errors="coerce")
    input_rows = len(data)
    duplicates = data.duplicated(subset=["time"] + price_qty_columns)
    duplicate_count = int(duplicates.sum())
    data = data.loc[~duplicates].copy()
    if data["time"].duplicated().any():
        raise ValueError("Conflicting same-time snapshots: verify an update sequence before proceeding")
    source_reversals = int((data["time"].diff() < pd.Timedelta(0)).sum())
    data = data.sort_values("time", kind="stable")
    bp, bq, ap, aq = [data[columns[key]].to_numpy(float)
                      for key in ("bid_price", "bid_qty", "ask_price", "ask_qty")]
    finite = np.isfinite(np.column_stack([bp,bq,ap,aq])).all(axis=1)
    positive_prices = (bp > 0).all(axis=1) & (ap > 0).all(axis=1)
    valid_qty = (bq >= 0).all(axis=1) & (aq >= 0).all(axis=1)
    ordered = (np.diff(bp, axis=1) < 0).all(axis=1) & (np.diff(ap, axis=1) > 0).all(axis=1)
    uncrossed = bp[:, 0] <= ap[:, 0]
    bid_depth, ask_depth = bq.sum(axis=1), aq.sum(axis=1)
    total = bid_depth + ask_depth
    valid = finite & positive_prices & valid_qty & ordered & uncrossed & (total > 0)
    imbalance = np.full(len(data), np.nan)
    imbalance[valid] = (bid_depth[valid] - ask_depth[valid]) / total[valid]
    result = pd.DataFrame({"time": data["time"].to_numpy(), "obi": imbalance,
                           "valid_snapshot": valid})
    audit = {"input_rows": input_rows, "exact_duplicates_removed": duplicate_count,
             "source_time_reversals": source_reversals, "invalid_snapshots": int((~valid).sum()),
             "first_timestamp": str(data["time"].min()), "last_timestamp": str(data["time"].max())}
    # Invalid snapshots are retained so they invalidate the book until a valid update arrives.
    return result, audit


snapshots = None
if READY:
    if SYMBOL == "UNCONFIRMED" or TIMESTAMP_MEANING == "UNCONFIRMED":
        raise ValueError("Confirm instrument and timestamp meaning before analysis")
    selected = [TIMESTAMP_COLUMN] + [template.format(level=i)
                for template in COLUMN_TEMPLATES.values() for i in range(1, K+1)]
    raw = pd.concat([pd.read_csv(path, sep=SEPARATOR, encoding=ENCODING,
                              usecols=selected) for path in files], ignore_index=True)
    snapshots, snapshot_audit = normalize_snapshots(
        raw, TIMESTAMP_COLUMN, TIMESTAMP_UNIT, K, COLUMN_TEMPLATES)
    display(pd.Series(snapshot_audit, name="Snapshot audit"))
    display(snapshots.head())
else:
    print("Snapshot validation pending actual data and field mapping.")

### 3.3 Sample without future information and split by time

At a grid timestamp, I use only snapshots with source timestamps **strictly earlier** than that timestamp. A snapshot older than the configured maximum age is unavailable. This is a freshness rule, not an execution-latency model.

Input is I_t and target is I_(t+1). Missing observations are never compressed away. Training, validation, and test cutoffs depend on elapsed time, not the number of surviving rows. A pair is included only when both endpoints fall inside the same split.

The default 60/20/20 split is an initial design. Very short files cannot establish reliable out-of-sample evidence. Review coverage and dates before proceeding.

In [ ]:
def make_grid(snapshots, interval, max_age):
    """Use strict backward matching on the original wall-clock grid."""
    step = pd.Timedelta(interval)
    if step <= pd.Timedelta(0) or pd.Timedelta(max_age) <= pd.Timedelta(0):
        raise ValueError("Sampling interval and maximum age must be positive")
    source = snapshots.sort_values("time").rename(columns={"time": "source_time"})
    times = pd.date_range(source.source_time.min().ceil(interval),
                          source.source_time.max().floor(interval), freq=interval)
    if len(times) < 10:
        raise ValueError("Too little time coverage for this sampling interval")
    grid = pd.merge_asof(pd.DataFrame({"time": times}), source,
                         left_on="time", right_on="source_time", direction="backward",
                         allow_exact_matches=False, tolerance=pd.Timedelta(max_age))
    grid["age_ms"] = (grid.time-grid.source_time).dt.total_seconds()*1000
    return grid.set_index("time")


def make_pairs(grid, train_fraction, validation_fraction, interval):
    """Keep unavailable pairs as NaNs; retain the full equal-spaced index."""
    if not (0 < train_fraction < 1 and 0 < validation_fraction < 1-train_fraction):
        raise ValueError("Invalid split fractions")
    frame = pd.DataFrame({"x": grid.obi, "y": grid.obi.shift(-1)}, index=grid.index)
    frame["target_time"] = frame.index + pd.Timedelta(interval)
    start, stop = frame.index.min(), frame.index.max()+pd.Timedelta(interval)
    duration = stop-start
    train_end = start + duration*train_fraction
    validation_end = start + duration*(train_fraction+validation_fraction)
    frame["split"] = "boundary"
    for name, lo, hi in [("train",start,train_end),("validation",train_end,validation_end),
                          ("test",validation_end,stop)]:
        mask = (frame.index >= lo) & (frame.index < hi) & (frame.target_time < hi)
        frame.loc[mask,"split"] = name
    return frame, {"start": str(start), "train_end": str(train_end),
                   "validation_end": str(validation_end), "stop": str(stop)}


pairs = None
if READY:
    grid = make_grid(snapshots, SAMPLE_INTERVAL, MAX_SNAPSHOT_AGE)
    pairs, cutoffs = make_pairs(grid, TRAIN_FRACTION, VALIDATION_FRACTION, SAMPLE_INTERVAL)
    coverage = pairs.assign(valid=pairs[["x","y"]].notna().all(axis=1)).groupby("split").agg(
        grid_pairs=("valid","size"), valid_pairs=("valid","sum"))
    coverage["valid_fraction"] = coverage.valid_pairs/coverage.grid_pairs
    display(pd.Series(cutoffs, name="UTC split boundaries"))
    display(coverage)
    if (coverage.loc[["train","validation","test"],"valid_fraction"] < 0.95).any():
        raise ValueError("A split has <95% valid pairs. Investigate gaps before proceeding.")
    display(grid.groupby(grid.index.floor("D")).agg(
        grid_rows=("obi","size"), valid_rows=("obi","count")))
else:
    print("Sampling and chronological splits pending real data.")

## 4. Hypothesis testing
### 4.1 Fit AR(1) and report robust coefficient uncertainty

The model minimizes training squared errors with an intercept. The mean baseline uses training target values only. The persistence baseline predicts I_(t+1) = I_t.

HAC means heteroskedasticity and autocorrelation consistent covariance estimation. The implementation uses Bartlett weights and the actual grid lag positions; missing rows have zero score contributions and do not collapse time. Inference is asymptotic and depends on regularity assumptions. It does not prove stationarity, causality, or model adequacy.

If the training sample is small relative to the selected HAC lag, the code stops instead of presenting a precise-looking result.

In [ ]:
def fit_ar1(frame, hac_lags=100, alpha=0.05):
    """Fit OLS and compute a grid-aware Newey-West/Bartlett covariance."""
    valid = frame[["x","y"]].notna().all(axis=1).to_numpy()
    n = int(valid.sum())
    if n < max(100, 10*(hac_lags+1)):
        raise ValueError("Insufficient training observations for the configured HAC lag")
    x = frame.loc[valid,"x"].to_numpy(float)
    y = frame.loc[valid,"y"].to_numpy(float)
    X = np.column_stack([np.ones(n), x])
    if np.linalg.matrix_rank(X) < 2:
        raise ValueError("Constant imbalance: the AR coefficient is unidentified")
    beta = np.linalg.lstsq(X,y,rcond=None)[0]
    residual = y-X@beta
    scores = np.zeros((len(frame),2))
    scores[valid] = X*residual[:,None]
    meat = scores.T@scores
    for lag in range(1, min(hac_lags,len(frame)-1)+1):
        cross = scores[lag:].T@scores[:-lag]
        meat += (1-lag/(hac_lags+1))*(cross+cross.T)
    bread = np.linalg.inv(X.T@X)
    covariance = bread@meat@bread*n/(n-2)
    se = np.sqrt(np.maximum(np.diag(covariance),0))
    z = np.divide(beta,se,out=np.full(2,np.nan),where=se>0)
    critical = norm.ppf(1-alpha/2)
    table = pd.DataFrame({"estimate":beta,"HAC_SE":se,"CI_low":beta-critical*se,
                          "CI_high":beta+critical*se,"p_two_sided":2*norm.sf(np.abs(z))},
                         index=["intercept","phi"])
    return {"beta":beta,"train_mean":float(y.mean()),"table":table,"n":n}


def predict(frame, model):
    """Return fixed predictions without clipping or substituting actual targets."""
    result = frame.copy()
    result["ar1"] = model["beta"][0]+model["beta"][1]*result.x
    result["mean"] = model["train_mean"]
    result["persistence"] = result.x
    result["residual"] = result.y-result.ar1
    return result


model = None
if READY:
    train = pairs.loc[pairs.split.eq("train")].copy()
    model = fit_ar1(train,HAC_LAGS,ALPHA)
    display(model["table"])
    if abs(model["beta"][1]) >= 1:
        warnings.warn("Estimated |phi| >= 1: a stationary AR(1) interpretation is unsupported.")
else:
    print("No coefficients estimated: real data have not been loaded.")

### 4.2 Diagnose dependence on the training segment

The empirical ACF below is the Pearson correlation of valid pairs at each physical lag. Pair counts are retained because missing data change the available information. This pairwise estimator may differ slightly from the conventional common-denominator ACF.

The theoretical AR(1) curve is a model implication, not an observed series. Large differences or remaining residual correlation indicate that the simple model may miss structure. I do not use a white-noise confidence band as a universal AR(1) goodness-of-fit test.

In [ ]:
def physical_acf(series, max_lag):
    """Compute lag correlations without compressing missing time points."""
    rows = []
    for lag in range(max_lag+1):
        aligned = pd.concat([series,series.shift(lag)],axis=1).dropna()
        value = aligned.iloc[:,0].corr(aligned.iloc[:,1]) if len(aligned)>2 else np.nan
        rows.append({"lag":lag,"acf":value,"pair_count":len(aligned)})
    return pd.DataFrame(rows).set_index("lag")


if READY:
    train_predictions = predict(train,model)
    observed_acf = physical_acf(train_predictions.x,ACF_MAX_LAG)
    residual_acf = physical_acf(train_predictions.residual,ACF_MAX_LAG)
    fig, axes = plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
    axes[0].plot(observed_acf.index,observed_acf.acf,"o-",label="Observed training ACF")
    if abs(model["beta"][1]) < 1:
        axes[0].plot(observed_acf.index,model["beta"][1]**observed_acf.index.to_numpy(),
                     "--",label="AR(1)-implied ACF")
    axes[0].set_title("Imbalance dependence")
    axes[1].plot(residual_acf.index,residual_acf.acf,"o-",label="Training residual ACF")
    axes[1].set_title("Unexplained linear dependence")
    for ax in axes:
        ax.axhline(0,color="gray",linewidth=0.8)
        ax.set(xlabel="Lag (sampling intervals)",ylabel="Correlation",ylim=(-1.05,1.05))
        ax.legend()
    plt.show()
    display(observed_acf.join(residual_acf,lsuffix="_observed",rsuffix="_residual"))

### 4.3 Evaluate validation predictions first

All models are compared on exactly the same available observations. MSE is mean squared error; MAE is mean absolute error. Imbalance is dimensionless, so these errors are dimensionless too. The out-of-range fraction records predictions outside [−1,1].

A Diebold-Mariano (DM) test compares models through the loss differential d = L(benchmark) − L(challenger), with L the squared error. Positive mean d favours the challenger. Its long-run variance uses Bartlett weights over physical grid lags. Missing grid points contribute zero and do not shift time. The primary comparison is AR(1) against the training mean; AR(1) against persistence is secondary. Periods where the book does not change make loss differentials persistent, so the bandwidth is wide (`DM_HAC_LAGS`), and other bandwidths are shown as sensitivity rather than chosen after the fact.

Validation results inform development. The final test is opened only by setting `RUN_FINAL_TEST = True` after freezing choices. Repeatedly changing the model after viewing test results turns that period into development data.

In [ ]:
def evaluate(predictions):
    valid = predictions[["x","y","ar1","mean","persistence"]].notna().all(axis=1)
    p = predictions.loc[valid]
    if len(p)<20:
        raise ValueError("Too few paired evaluation observations")
    rows=[]
    for name in ["ar1","mean","persistence"]:
        error=p.y-p[name]
        rows.append({"model":name,"N":len(p),"MSE":float((error**2).mean()),
                     "MAE":float(error.abs().mean()),
                     "out_of_range_fraction":float((p[name].abs()>1).mean())})
    metrics=pd.DataFrame(rows).set_index("model")
    base=metrics.loc["mean","MSE"]
    metrics["improvement_vs_mean"] = 1-metrics.MSE/base if base>0 else np.nan
    return metrics



def dm_test(predictions, challenger, benchmark, hac_lags):
    """Two-sided DM test of equal squared-error loss with a grid-aware Bartlett HAC variance."""
    d = ((predictions.y-predictions[benchmark])**2-(predictions.y-predictions[challenger])**2)
    valid = d.notna().to_numpy()
    n = int(valid.sum())
    if n < 10*(hac_lags+1):
        raise ValueError("Too few loss differentials for the configured DM bandwidth")
    mean_d = float(d[valid].mean())
    u = np.zeros(len(d))
    u[valid] = d.to_numpy()[valid]-mean_d
    long_run = float(u@u)/n
    for lag in range(1, hac_lags+1):
        long_run += 2*(1-lag/(hac_lags+1))*float(u[lag:]@u[:-lag])/n
    se = np.sqrt(max(long_run,0)/n)
    stat = mean_d/se if se>0 else np.nan
    return {"challenger":challenger,"benchmark":benchmark,"HAC_lags":hac_lags,"N":n,
            "mean_loss_diff":mean_d,"HAC_SE":se,"DM_stat":stat,"p_two_sided":float(2*norm.sf(abs(stat))),
            "relative_MSE_reduction":mean_d/float(((predictions.y-predictions[benchmark])**2)[valid].mean())}


def dm_table(predictions):
    rows=[dm_test(predictions,c,b,lag) for c,b in DM_COMPARISONS
          for lag in (DM_HAC_LAGS,)+tuple(DM_SENSITIVITY_LAGS)]
    table=pd.DataFrame(rows)
    table["role"]=np.where(table.HAC_lags.eq(DM_HAC_LAGS),"primary_bandwidth","sensitivity")
    return table.set_index(["challenger","benchmark","HAC_lags"])


def daily_metrics(predictions):
    p=predictions.dropna(subset=["x","y"])
    out={name:((p.y-p[name])**2).groupby(p.index.floor("D")).mean() for name in ["ar1","mean","persistence"]}
    frame=pd.DataFrame(out).add_prefix("MSE_")
    frame["N"]=p.groupby(p.index.floor("D")).size()
    frame["unchanged_fraction"]=(p.y==p.x).groupby(p.index.floor("D")).mean()
    return frame

validation_predictions = None
validation_dm = None
if READY:
    validation_predictions=predict(pairs.loc[pairs.split.eq("validation")],model)
    display(evaluate(validation_predictions))
    validation_dm=dm_table(validation_predictions)
    display(validation_dm)
    display(daily_metrics(validation_predictions))
    # Only the first 200 grid positions are shown; no realized values are regenerated.
    view=validation_predictions.iloc[:200]
    ax=view[["y","ar1","mean","persistence"]].plot(figsize=(12,4),ylim=(-1.1,1.1))
    ax.set(title="Validation: fixed observations and predictions",xlabel="Prediction origin (UTC)",
           ylabel="Next-period imbalance")
    plt.show()
    print("These are validation results, not final-test evidence.")

if READY and RUN_FINAL_TEST:
    test_predictions=predict(pairs.loc[pairs.split.eq("test")],model)
    test_metrics=evaluate(test_predictions)
    display(test_metrics)
    test_dm=dm_table(test_predictions)
    display(test_dm)
    daily=daily_metrics(test_predictions)
    display(daily)
    print("Final test: model, bandwidth, and comparisons were fixed before this cell was enabled.")
elif READY:
    print("Final test remains unopened. Freeze the analysis before enabling RUN_FINAL_TEST.")

### 4.4 Optional interactive model comparison

The slider changes a candidate φ while holding the fitted intercept, validation inputs, and realized targets fixed. This shows what a different persistence assumption would predict on the **same data**. It does not refit the official model or modify reported metrics. Any model selected through this interaction is a validation-stage choice.

This cell is optional and requires `ipywidgets`. The actual observations never move when the slider changes.

In [ ]:
if READY:
    try:
        import ipywidgets as widgets
    except ImportError:
        print("Optional: install ipywidgets to enable the candidate-coefficient slider.")
    else:
        fixed_view=validation_predictions.iloc[:200].copy(deep=True)
        def compare_candidate(phi):
            fig,ax=plt.subplots(figsize=(12,4))
            ax.plot(fixed_view.index,fixed_view.y,label="Realized next imbalance",color="black")
            candidate=model["beta"][0]+phi*fixed_view.x
            ax.plot(fixed_view.index,candidate,label="Candidate prediction",alpha=0.8)
            ax.set(xlabel="Prediction origin (UTC)",ylabel="Imbalance",
                   title="Validation exploration: realized values stay fixed",ylim=(-1.2,1.2))
            ax.legend()
            plt.show()
        display(widgets.interactive(compare_candidate,
            phi=widgets.FloatSlider(value=float(np.clip(model["beta"][1],-1.1,1.1)),
                                    min=-1.1,max=1.1,step=0.01,description="Candidate φ",
                                    continuous_update=False)))

### 4.5 Save reproducibility details

Exports include the actual data file hashes, environment versions, schema mapping, sampling rules, split boundaries, and fitted coefficients. Hashing large files reads them fully and may take time. Raw market data are not copied into the report.

Forecast improvements are assessed with the DM tests in §4.3. They are dependence-aware but asymptotic, and they depend on the chosen bandwidth. The sensitivity rows show how much.

In [ ]:
def sha256_file(path):
    """Hash bytes in bounded chunks rather than loading the entire file."""
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda:handle.read(8*1024*1024),b""):
            digest.update(block)
    return digest.hexdigest()


if READY:
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    manifest={"symbol":SYMBOL,"files":[{"path":str(p),"sha256":sha256_file(p)} for p in files],
              "timestamp_column":TIMESTAMP_COLUMN,"timestamp_unit":TIMESTAMP_UNIT,
              "timestamp_meaning":TIMESTAMP_MEANING,"column_templates":COLUMN_TEMPLATES,
              "K":K,"interval":SAMPLE_INTERVAL,"max_age":MAX_SNAPSHOT_AGE,
              "cutoffs":cutoffs,"HAC_lags":HAC_LAGS,"alpha":ALPHA,"DM_HAC_lags":DM_HAC_LAGS,
              "DM_sensitivity_lags":list(DM_SENSITIVITY_LAGS),"DM_comparisons":DM_COMPARISONS,
              "final_test_opened":RUN_FINAL_TEST,"snapshot_audit":snapshot_audit,
              "python":platform.python_version(),"numpy":np.__version__,
              "pandas":pd.__version__,"scipy":scipy.__version__}
    (OUTPUT_DIR/"manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    model["table"].to_csv(OUTPUT_DIR/"coefficients.csv")
    evaluate(validation_predictions).to_csv(OUTPUT_DIR/"validation_metrics.csv")
    validation_dm.to_csv(OUTPUT_DIR/"validation_dm.csv")
    if RUN_FINAL_TEST:
        test_metrics.to_csv(OUTPUT_DIR/"test_metrics.csv")
        test_dm.to_csv(OUTPUT_DIR/"test_dm.csv")
        daily.to_csv(OUTPUT_DIR/"test_metrics_by_day.csv")
    print("Saved analysis outputs to",OUTPUT_DIR.resolve())
else:
    print("Nothing exported: no market data have been analyzed.")

## 5. Conclusion

Results from the run on 2026-11-03. The settings were frozen before the test set was opened; see `reports/2026-11-03_freeze_record.md`. Exported tables are in `results/ar1_obi/`.

- **Data:** `binance_spot` ETHUSD spot, 2026-10-10 to 2026-10-19 UTC. The book was rebuilt from local incremental L2 logs with 0 sequence gaps, 0 crossed books, and 0 BBO mismatches. K = 5, 1 s grid, local receive time, maximum book age 300 s. Valid pairs: 100% in every split.
- **Persistence:** on the training data (518,401 pairs), φ = 0.949 (HAC 95% CI 0.948–0.951) and c = −0.0038. The implied long-run mean is about −0.074. Imbalance is highly persistent at 1 s. Part of this is mechanical: in the test period, 72–74% of adjacent grid points have an unchanged book.
- **Model adequacy:** the residual ACF is −0.16 at lag 1 and −0.03 at lag 2, then within ±0.02. AR(1) leaves short-lag dependence unexplained. The negative lag-1 residual correlation is consistent with a mixture of unchanged periods and partial reversal after book changes, rather than one geometric decay.
- **Prediction (test, 172,799 pairs):** MSE is 0.00417 for AR(1), 0.04254 for the training mean, and 0.00427 for persistence.
  - Primary DM test (AR(1) vs mean, 600 lags): 90.2% MSE reduction, DM = 14.9, p < 1e−49.
  - Secondary test (AR(1) vs persistence): 2.5% MSE reduction, DM = 11.1, p < 1e−28.
  - Both stay significant at 60 and 3600 lags.
  - By MAE, AR(1) is *worse* than persistence (0.0284 vs 0.0227). Its edge is limited to squared error: it shrinks toward the mean, which helps on large moves but costs accuracy on the many unchanged intervals.
- **Stability:** AR(1) beat persistence on both test days, and on validation day 10-16, but not on validation day 10-17 (MSE 0.00451 vs 0.00443). That was the quietest day, with 80% of intervals unchanged. The gain over persistence is small and depends on the day.
- **Uncertainty:** DM inference is asymptotic, and its p-values depend on the bandwidth. The very small p-values reflect the large N. They do not indicate a large effect. With 10 days of data, this does not cover different market regimes.
- **Boundaries:** these results show only that the next 1 s imbalance is predictable. They say nothing about returns, mid-price changes, or profitability after costs.

### Before extending the study

A different sampling interval, number of levels, or transformation is a new modeling choice. Record changes and preserve a genuinely unseen evaluation period. Changes to the raw data require rerunning `scripts/clean_l2.py` and reviewing `cleaning_audit.json`.